In [1]:
import os          
import pathlib     
                   
import sys         

here = pathlib.Path.cwd()      

ROOT = here.parents[2] if here.name == "day05" else here
os.chdir(ROOT)                 

SANDBOX = ROOT / "sandbox" / "w2" / "day05"

BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

# 리포지토리

In [2]:

from datetime import date
from app.db.session import get_sessionmaker, get_engine
from app.models import Department, Document, DocumentVersion
from app.db.init_db import init_db

# 연습용 DB경로
연습DB = SANDBOX / "repo_practice.db"
연습DB.unlink(missing_ok=True)           # 파일이 없어도 오류 내지 않는다.                          

# 연습용 DB 환경 설정
연습엔진 = get_engine(f"sqlite:///{연습DB}")
init_db(연습엔진)                   
연습세션 = get_sessionmaker(연습엔진)  

# SQL 연습
with 연습세션() as s:
    # 부서 저장
    s.add_all([
        Department(id="HRGA", name="인사총무"),
        Department(id="PU", name="구매팀"),
        Department(id="SE", name="보안팀"),
    ])
    s.flush()  # 쿼리문 나감 -> DB저장
    # 문서 저장    
    s.add_all([
        Document(id="DOC-HR-014", title="국내출장 여비 규정",
                 dept_id="HRGA", security_level="일반"),
        Document(id="DOC-PU-007", title="구매·계약 규정",
                 dept_id="PU", security_level="대외비"),
        Document(id="DOC-SE-003", title="정보보안 지침",
                 dept_id="SE", security_level="대외비"),
    ])
    s.flush()
    s.add_all([
        DocumentVersion(doc_id="DOC-HR-014", version="v2.0", status="현행",
                        effective_from=date(2025, 7, 1), expires_at=None,
                        file_path="uploads/DOC-HR-014_v2.0.docx", file_format="docx"),
        DocumentVersion(doc_id="DOC-PU-007", version="v4.0", status="현행",
                        effective_from=date(2026, 3, 1), expires_at=None,
                        file_path="uploads/DOC-PU-007_v4.0.pdf", file_format="pdf"),
        DocumentVersion(doc_id="DOC-SE-003", version="v2.2", status="현행",
                        effective_from=date(2025, 10, 1), expires_at=None,
                        file_path="uploads/DOC-SE-003_v2.2.pdf", file_format="pdf"),
    ])
    # DB에 영구 저장
    s.commit()

print("연습 DB   :", 연습DB.relative_to(ROOT))
print("문서 3건 · 버전 3건 준비 완료 (대외비 2건 포함)")


연습 DB   : sandbox/w2/day05/repo_practice.db
문서 3건 · 버전 3건 준비 완료 (대외비 2건 포함)


In [ ]:
from sqlalchemy import select

# 쿼리문 SQLAlchemy 2.x 버전으로 생성
stmt = select(Document).where(Document.dept_id == "HRGA")

print(stmt)                                  
print()
print("바인딩된 값 :", stmt.compile().params)